# 05 · Consumer groups, parallelism and rebalancing
**Level:** intermediate · **Time:** 45 minutes · **Prerequisite:** 04.
Start two real consumers in one group, inspect ownership, stop a member, and observe
the remaining member take over. This lesson uses the classic group protocol and eager
`range` assignor explicitly; cooperative and the newer consumer protocol differ.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


In [ ]:
require_broker()
topic = new_topic('groups', partitions=3)


## 1. Capture rebalance callbacks
We poll both consumers from one notebook thread for a small bounded demonstration.
In services, give each worker its own Consumer and polling loop. Callbacks show assignment
changes; the default client action applies the assignment after the callback returns.


In [ ]:
from time import monotonic
events = []
def assigned(label):
    def callback(c, partitions):
        events.append((label, "assigned", [p.partition for p in partitions]))
    return callback
def revoked(label):
    def callback(c, partitions):
        events.append((label, "revoked", [p.partition for p in partitions]))
    return callback
group = f"lab.team.{RUN}"
group_config = {"group.protocol": "classic", "partition.assignment.strategy": "range",
                "session.timeout.ms": 6000, "heartbeat.interval.ms": 2000}


## 2. Wait for stable assignments before producing
The polling here is not processing business events: the fresh topic is still empty.
The assertions require all three partitions to be split without overlap.


In [ ]:
c1 = consumer(group, **group_config)
c2 = consumer(group, **group_config)
try:
    c1.subscribe([topic], on_assign=assigned("A"), on_revoke=revoked("A"))
    c2.subscribe([topic], on_assign=assigned("B"), on_revoke=revoked("B"))
    deadline = monotonic() + 30
    while monotonic() < deadline:
        c1.poll(0.1)
        c2.poll(0.1)
        a = {p.partition for p in c1.assignment()}
        b = {p.partition for p in c2.assignment()}
        if a and b and not a & b and a | b == {0, 1, 2}:
            break
    else:
        raise TimeoutError("Group did not settle; inspect broker/group logs.")
    print("A owns", sorted(a), "B owns", sorted(b))

    # Produce to every partition so both members definitely have work.
    import json
    p = producer()
    delivery_errors = []
    def report(error, message):
        if error:
            delivery_errors.append(str(error))
    for partition in range(3):
        for number in range(3):
            p.produce(topic, partition=partition,
                      value=json.dumps({"id": f"{partition}-{number}"}).encode(), on_delivery=report)
    assert p.flush(15) == 0 and not delivery_errors, delivery_errors
    seen = {"A": [], "B": []}
    deadline = monotonic() + 20
    while sum(map(len, seen.values())) < 9 and monotonic() < deadline:
        for label, c in [("A", c1), ("B", c2)]:
            m = c.poll(0.1)
            if m is None:
                continue
            if m.error():
                raise RuntimeError(m.error())
            seen[label].append((m.partition(), decode(m)["id"]))
            c.commit(message=m, asynchronous=False)
    ids = [event_id for rows in seen.values() for _, event_id in rows]
    assert len(ids) == len(set(ids)) == 9, seen
    print("Work by member:", seen)

    # Graceful departure triggers reassignment; no processing is in flight.
    c2.close()
    c2 = None
    deadline = monotonic() + 30
    while monotonic() < deadline:
        c1.poll(0.2)
        if {p.partition for p in c1.assignment()} == {0, 1, 2}:
            break
    else:
        raise TimeoutError("Remaining member did not acquire all partitions.")
    print("After B leaves, A owns:", [p.partition for p in c1.assignment()])
finally:
    c1.close()
    if c2 is not None:
        c2.close()


## 3. Inspect the timeline
There can be more than one assignment callback during startup. A rebalance changes
ownership; it does not erase stored offsets. This clean run has no duplicate work,
but failures between processing and commit can still cause duplicates.


In [ ]:
for entry in events:
    print(entry)


## 4. An independent group sees the complete stream
This is useful for analytics and audit consumers that must not share work with each other.


In [ ]:
audit = collect(topic, count=9, group_id=f"lab.group-audit.{RUN}")
assert {decode(m)["id"] for m in audit} == {f"{partition}-{number}" for partition in range(3) for number in range(3)}
print("Audit group read", len(audit), "records")


## 5. Reason about scaling
| Partitions | Active group members | Maximum members with assignments |
|---:|---:|---:|
| 3 | 1 | 1 |
| 3 | 2 | 2 |
| 3 | 5 | 3 |

Increasing members beyond partition count leaves members idle in a traditional consumer
group. Slow processing can exceed `max.poll.interval.ms`, trigger reassignment and make
old work unsafe to commit. Increasing that timeout hides symptoms if the underlying
work is unbounded. Monitor lag, handler duration and rebalance frequency together.

## Exercise
Change the topic to two partitions and add a third consumer. Poll all three and print
their assignments. Then stop the member owning a partition. What changes?
<details><summary>Solution hints</summary>
One member has no partitions. Close the chosen consumer, continue polling the others,
and wait with a deadline for the two partitions to be reassigned. Keep `try/finally`
around all clients. Do not copy the polling-only assignment loop onto a nonempty topic:
it would discard fetched records without processing them.
</details>
Next: **06 — event contracts and schema evolution**.
